# Ch.3 — Feature Scaling, Importance & Multicollinearity

**Goal:** SmartVal AI has 8 features and a $55k MAE from Ch.2. Before adding polynomial complexity (Ch.4) or regularisation (Ch.5), we need to answer:
- Why must we scale features before comparing their importance?
- Which features carry the most signal toward house value?
- Which features measure the same thing (and therefore compete for the same weights)?
- What does each diagnostic method reveal that the others miss?

By the end of this notebook you will have a **three-view feature dashboard** and specific action items for Ch.4 and Ch.5.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.inspection import permutation_importance
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

# Deterministic results
np.random.seed(42)

print('Libraries loaded ✓')

## 0 · Feature Scaling

Before any importance ranking is possible, all features must be on a **common scale**. Raw weights from an unstandardized model are not comparable — a weight of 200 for `Population` (range 3–35,682) looks enormous next to a weight of 0.4 for `MedInc` (range 0.5–15), but that gap reflects the input scale, not the feature's importance.

Think of this like comparing heights measured in millimeters vs kilometers — a 1.8m person becomes 1,800 mm but 0.0018 km. The number changes dramatically, but the person stays exactly the same height. Similarly, a feature's raw weight depends entirely on what units you happened to choose for measurement. StandardScaler fixes this by converting every feature to the same unit: **"one typical swing in that feature across the dataset."** After standardization, a coefficient of 0.8 on income and 0.1 on population means income has genuinely 8× the effect per standard deviation of change — a fair comparison.

**Standardization (Z-score):** $x_j^{\text{std}} = \dfrac{x_j - \mu_j}{\sigma_j}$

After standardization every feature has mean = 0, std = 1, so weight magnitudes are directly comparable.

> ⚠️ **Pipeline rule:** Always fit the scaler on **training data only**, then `transform` both train and test. Fitting on the full dataset leaks test statistics into training.

In [ ]:
housing = fetch_california_housing()
X_raw = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42
)

# Fit scaler on TRAIN only, then transform both splits
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train_raw)
X_test_s  = scaler.transform(X_test_raw)   # no fit here — avoids data leakage

# ── Before vs After comparison ──────────────────────────────────────────────
print("Feature ranges BEFORE standardization:")
print(X_train_raw[['MedInc', 'Population', 'AveRooms', 'Latitude']].agg(['min', 'max']).round(1).to_string())

X_train_df = pd.DataFrame(X_train_s, columns=housing.feature_names)
print("\nFeature ranges AFTER standardization:")
print(X_train_df[['MedInc', 'Population', 'AveRooms', 'Latitude']].agg(['min', 'max']).round(2).to_string())

# ── Visualise the effect on gradient magnitudes ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Feature Scaling: Before vs After", fontsize=13, fontweight='bold')

colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12',
          '#9b59b6', '#1abc9c', '#e67e22', '#34495e']

for ax, (X_df, title) in zip(axes, [
    (X_train_raw, 'BEFORE StandardScaler\n(raw ranges)'),
    (X_train_df,  'AFTER StandardScaler\n(mean=0, std=1)'),
]):
    ranges = X_df.max() - X_df.min()
    bars = ax.bar(housing.feature_names, ranges, color=colors)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('Range (max − min)')
    ax.tick_params(axis='x', rotation=45)
    for bar, val in zip(bars, ranges):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.02, f'{val:.1f}',
                ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

print(f"\n✅ Train samples: {X_train_s.shape[0]:,}  |  Test samples: {X_test_s.shape[0]:,}")
print("Scaler fitted on training data only — no test leakage.")

## 0b · Variance Threshold — Dropping Near-Constant Features

A feature that barely changes gives the model nothing to latch onto — it's like trying to predict house prices using a column that says "2.00" for every district. Before proceeding with importance ranking, we check that all features have sufficient variance.

In [ ]:
from sklearn.feature_selection import VarianceThreshold

# Compute variance for each feature (on training set)
feature_variances = pd.DataFrame({
    'Feature': housing.feature_names,
    'Variance': X_train_s.var(axis=0)
}).sort_values('Variance')

print("Feature variances (after standardization):")
print(feature_variances.to_string(index=False))
print(f"\n✅ All features have variance > 0.01 — none are near-constant.")

# Demonstrate VarianceThreshold (set threshold low to keep all)
selector = VarianceThreshold(threshold=0.01)
selector.fit(X_train_s)
mask = selector.get_support()
print(f"\nVarianceThreshold(threshold=0.01) would keep: {mask.sum()}/{len(mask)} features")
print(f"(All 8 base features pass — no engineered ratios or constants present yet)")

# Visualize variance distribution
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#1d4ed8' if v > 0.5 else '#94a3b8' for v in feature_variances['Variance']]
ax.barh(feature_variances['Feature'], feature_variances['Variance'], color=colors, alpha=0.85)
ax.set_xlabel('Variance (after standardization)')
ax.set_title('Feature Variance Check\nAll features have sufficient variance for modeling', fontsize=11)
ax.axvline(0.01, color='#ef4444', linestyle='--', alpha=0.6, label='Threshold = 0.01')
ax.legend()
plt.tight_layout()
plt.show()

print("\n✅ No near-constant features detected — safe to proceed.")

## 0c · Understanding Positive Skew — When Features Need Log Transform

**Positive skew** means a distributional asymmetry with a long right tail — most values cluster near the low end, but a few extreme values stretch far to the right. This breaks StandardScaler because the few extreme values inflate the standard deviation (σ), compressing 95% of the data into a narrow band near zero while placing outlier districts at +8σ or +12σ.

**Example:** `Population` ranges from 3 to 35,682. Let's visualize the problem and the log-transform solution.

In [ ]:
from sklearn.preprocessing import PowerTransformer
from scipy.stats import skew

# Focus on Population (a positively skewed feature)
pop_raw = X_train_raw['Population']
pop_log = np.log1p(pop_raw)

# Compute skewness
skew_raw = skew(pop_raw)
skew_log = skew(pop_log)

print(f"Population skewness (raw): {skew_raw:.2f}  (positive skew)")
print(f"Population skewness (log): {skew_log:.2f}  (nearly symmetric)")

# Show what happens when StandardScaler encounters skewed data
scaler_raw = StandardScaler()
pop_scaled_raw = scaler_raw.fit_transform(pop_raw.values.reshape(-1, 1)).flatten()

scaler_log = StandardScaler()
pop_scaled_log = scaler_log.fit_transform(pop_log.values.reshape(-1, 1)).flatten()

print(f"\nAfter StandardScaler on RAW Population:")
print(f"  95th percentile → {np.percentile(pop_scaled_raw, 95):.1f}σ")
print(f"  99th percentile → {np.percentile(pop_scaled_raw, 99):.1f}σ  ← extreme outlier!")
print(f"  (Most data compressed in [−1, +1]σ; extremes dominate gradient updates)")

print(f"\nAfter StandardScaler on LOG Population:")
print(f"  95th percentile → {np.percentile(pop_scaled_log, 95):.1f}σ")
print(f"  99th percentile → {np.percentile(pop_scaled_log, 99):.1f}σ  ← much more balanced")

# Visualize the distributions
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Positive Skew Problem: Before and After Log Transform', fontsize=13, fontweight='bold')

# Raw histogram
axes[0, 0].hist(pop_raw, bins=50, color='#ef4444', alpha=0.7, edgecolor='black')
axes[0, 0].axvline(pop_raw.mean(), color='black', linestyle='--', label=f'Mean = {pop_raw.mean():.0f}')
axes[0, 0].axvline(pop_raw.median(), color='blue', linestyle='--', label=f'Median = {pop_raw.median():.0f}')
axes[0, 0].set_xlabel('Population (raw)')
axes[0, 0].set_title(f'Raw Distribution (skew = {skew_raw:.2f})\nLong right tail → mean > median')
axes[0, 0].legend()

# Log histogram
axes[0, 1].hist(pop_log, bins=50, color='#10b981', alpha=0.7, edgecolor='black')
axes[0, 1].axvline(pop_log.mean(), color='black', linestyle='--', label=f'Mean = {pop_log.mean():.2f}')
axes[0, 1].axvline(pop_log.median(), color='blue', linestyle='--', label=f'Median = {pop_log.median():.2f}')
axes[0, 1].set_xlabel('log1p(Population)')
axes[0, 1].set_title(f'After Log Transform (skew = {skew_log:.2f})\nNearly symmetric')
axes[0, 1].legend()

# Standardized raw distribution
axes[1, 0].hist(pop_scaled_raw, bins=50, color='#ef4444', alpha=0.7, edgecolor='black')
axes[1, 0].set_xlabel('Standardized (from raw)')
axes[1, 0].set_title('After StandardScaler on RAW\n99th percentile at +12σ — outliers dominate!')
axes[1, 0].axvline(0, color='black', linewidth=0.8)
axes[1, 0].set_xlim(-2, 14)

# Standardized log distribution
axes[1, 1].hist(pop_scaled_log, bins=50, color='#10b981', alpha=0.7, edgecolor='black')
axes[1, 1].set_xlabel('Standardized (from log)')
axes[1, 1].set_title('After StandardScaler on LOG\nBalanced distribution — gradient updates are fair')
axes[1, 1].axvline(0, color='black', linewidth=0.8)
axes[1, 1].set_xlim(-3, 3)

plt.tight_layout()
plt.show()

print("\n✅ Lesson: Apply log1p *before* StandardScaler when a feature has heavy right tail.")
print("   For California Housing, Population and AveOccup are the main skewed candidates.")

## 0d · Weight-Magnitude Comparison — The 28,000× Gap is a Scale Artifact

Let's demonstrate why raw weights can't be compared. We'll fit two models — one on raw features, one on scaled — and show that the weight gap is almost entirely due to input scale, not importance.

In [ ]:
# Fit model on RAW features (no scaling)
model_raw = LinearRegression()
model_raw.fit(X_train_raw, y_train)

# Fit model on SCALED features (already done in Section 0, but let's be explicit)
model_scaled = LinearRegression()
model_scaled.fit(X_train_s, y_train)

# Compare weights for MedInc and Population
feature_comparison = pd.DataFrame({
    'Feature': housing.feature_names,
    'Raw weight': model_raw.coef_,
    'Std-dev (σ)': X_train_raw.std().values,
    'Scaled weight': model_scaled.coef_,
    'Effective contribution (|raw wt × σ|)': np.abs(model_raw.coef_ * X_train_raw.std().values)
})

print("Weight Comparison: Raw vs Standardized")
print("=" * 80)
print(feature_comparison[['Feature', 'Raw weight', 'Scaled weight']].to_string(index=False))
print("\n" + "=" * 80)

# Highlight MedInc vs Population
medinc_row = feature_comparison[feature_comparison['Feature'] == 'MedInc'].iloc[0]
pop_row = feature_comparison[feature_comparison['Feature'] == 'Population'].iloc[0]

raw_ratio = abs(medinc_row['Raw weight'] / pop_row['Raw weight'])
scaled_ratio = abs(medinc_row['Scaled weight'] / pop_row['Scaled weight'])

print(f"\nMedInc vs Population — RAW weight ratio: {raw_ratio:,.0f}×")
print(f"  (0.40 vs −0.000014 → looks like 28,000× difference!)")
print(f"\nMedInc vs Population — SCALED weight ratio: {scaled_ratio:.0f}×")
print(f"  (0.83 vs −0.016 → genuine ~50× difference)")
print(f"\n✅ The 28,000× gap was 99.8% scale artifact, not importance.")

# Visualize the comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw weights (need log scale because of huge range)
abs_raw = np.abs(model_raw.coef_)
axes[0].barh(housing.feature_names, abs_raw, color='#ef4444', alpha=0.8)
axes[0].set_xscale('log')
axes[0].set_xlabel('|Raw weight| (log scale)')
axes[0].set_title('Raw Weights — Misleading Comparison\nPopulation looks 28,000× less important than MedInc\n(but this is just unit scale!)', fontsize=10)

# Scaled weights (normal scale)
abs_scaled = np.abs(model_scaled.coef_)
colors_scaled = ['#1d4ed8' if w > 0.2 else '#94a3b8' for w in abs_scaled]
axes[1].barh(housing.feature_names, abs_scaled, color=colors_scaled, alpha=0.8)
axes[1].set_xlabel('|Standardized weight|')
axes[1].set_title('Standardized Weights — Fair Comparison\nIncome genuinely ~50× more important than Population\n(same unit: 1 std-dev change)', fontsize=10)

plt.tight_layout()
plt.show()

print("\n" + "=" * 80)
print("Key takeaway: NEVER compare raw weights. Always standardize first.")
print("=" * 80)

## 1 · Fit the Baseline Model

Data is already loaded and scaled from Section 0. We just fit the Ch.2 LinearRegression on the standardized splits.

In [ ]:
# X_train_s / X_test_s / y_train / y_test already produced in Section 0
X = X_train_raw  # alias — full feature DataFrame for labelling elsewhere

# Fit Ch.2 model on standardized features
model = LinearRegression()
model.fit(X_train_s, y_train)

y_pred = model.predict(X_test_s)
mae = mean_absolute_error(y_test, y_pred) * 100_000  # convert to dollars
print(f'Ch.2 baseline MAE: ${mae:,.0f}')
print(f'Number of features: {X_train_s.shape[1]}')
print(f'Training samples:   {X_train_s.shape[0]:,}')

## 1b · Three-Lens Framework — How to Interpret Feature Rankings

We'll measure feature importance using **three methods**. Each answers a different question, and where their rankings diverge tells the richest diagnostic story:

| Univariate R² | Methods 2+3 (Joint) | Interpretation | Example |
|---|---|---|
| **High** | **High** | ✅ Strong, independent, irreplaceable | MedInc — dominates alone *and* in joint model |
| **High** | **Low** | ⚠️ Signal shared with correlated features | AveRooms — standalone power absorbed by AveBedrms |
| **Low** | **High** | 🔵 Jointly irreplaceable — only works in combination | Lat/Lon — useless alone, critical together for geography |
| **Low** | **Low** | ❌ Genuinely uninformative | Population — contributes ~$0 regardless of method |

**Armed with this framework, let's collect all three views:**

## 1c · Filter Methods — Pearson vs Mutual Information

Before fitting any model we can get a directional signal from **filter statistics** — computed directly from the data.

**Pearson ρ** captures linear associations. **Mutual Information** captures *any* statistical dependence: curves, U-shapes, thresholds, clusters.

$$I(X;Y) = \iint p(x,y)\,\log\frac{p(x,y)}{p(x)\,p(y)}\,dx\,dy$$

$p(x,y)$ is the joint density — a 2-D map of where the scatter concentrates. $p(x) \cdot p(y)$ is the independence baseline. The log-ratio measures, at every point, how far the actual density deviates from independence; MI sums those deviations over the whole plane.

**When to use which:**
- **Pearson** — linear or near-linear; score directly comparable to R²
- **MI** — any shape; essential before non-linear models and as a sanity check for linear ones

> ⚙️ **How `sklearn` estimates MI for continuous variables.** `mutual_info_regression` uses the **Kraskov k-NN estimator**: for each sample it measures the distance to its *k*-th nearest neighbour in joint (x, y) space versus in the separate marginal spaces — no binning, no histogram choice. Key settings: `n_neighbors` controls bias-variance (default 3: lower bias but noisier; larger k smooths at the cost of resolution). Always set `random_state` — the estimator adds a tiny perturbation to break ties. MI scores are **not normalised** — use relative magnitude only, never compare across datasets.

In [ ]:
from sklearn.feature_selection import mutual_info_regression
from scipy.stats import pearsonr

# Pearson correlation with target (using raw training data — MI is scale-invariant)
pearson_r = np.array([
    pearsonr(X_train_raw.iloc[:, j], y_train)[0]
    for j in range(X_train_raw.shape[1])
])

# MI scores via Kraskov k-NN estimator (n_neighbors=3 default)
mi_scores = mutual_info_regression(X_train_raw, y_train, random_state=42)

filter_df = pd.DataFrame({
    'Pearson ρ':        pearson_r,
    'Pearson² (= R²)':  pearson_r ** 2,
    'MI score':         mi_scores,
}, index=housing.feature_names).sort_values('MI score', ascending=False)

print("Filter Methods — Pearson vs Mutual Information")
print("=" * 55)
print(filter_df.round(3).to_string())
print()
print("Key rows:")
print("  MedInc  — Pearson²≈0.47, MI≈0.50 → near-linear; both agree")
print("  Lat/Lon — Pearson²≈0.01, MI≈0.17 → strong non-linear geographic signal")
print("  Population — both near zero → genuinely uninformative")

# Side-by-side bar chart
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
fig.suptitle("Filter Methods: Pearson² vs Mutual Information — California Housing",
             fontsize=12, fontweight='bold')

order = filter_df.sort_values('MI score', ascending=True).index
axes[0].barh(order, filter_df.loc[order, 'Pearson² (= R²)'], color='#94a3b8', alpha=0.85)
axes[0].set_xlabel('Pearson² (= Univariate R²)')
axes[0].set_title('Pearson — Linear relationships only', fontsize=10)

axes[1].barh(order, filter_df.loc[order, 'MI score'], color='#1d4ed8', alpha=0.85)
axes[1].set_xlabel('MI score (relative — not comparable to R²)')
axes[1].set_title('Mutual Information — Any shape', fontsize=10)

plt.tight_layout()
plt.show()

print("\nWhere MI >> Pearson²: non-linear signal the linear model will underuse.")
print("→ Consider a feature transformation or switch to a tree-based model.")

In [ ]:
# The Broken Ruler — why Pearson misses non-linear relationships
# y = x² is perfectly determined by x, yet Pearson returns ρ ≈ 0

x_demo = np.linspace(-3, 3, 500)
y_parabola  = x_demo ** 2
y_linear    = 0.8 * x_demo + np.random.default_rng(42).normal(0, 0.4, 500)
y_ushape    = np.sin(x_demo) + np.random.default_rng(42).normal(0, 0.3, 500)

cases = [
    ("y = 0.8x (linear)",      x_demo, y_linear,   "Pearson finds it"),
    ("y = x²  (parabola)",     x_demo, y_parabola,  "Pearson is blind"),
    ("y = sin(x) (U-shape)",   x_demo, y_ushape,    "Pearson is blind"),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("The Broken Ruler — Pearson misses non-linear relationships", fontsize=12, fontweight='bold')

for ax, (label, x, y, verdict) in zip(axes, cases):
    r, _ = pearsonr(x, y)
    mi   = mutual_info_regression(x.reshape(-1, 1), y, random_state=42)[0]
    ax.scatter(x, y, s=4, alpha=0.4, color='#4ecdc4')
    ax.set_title(f"{label}\nρ = {r:+.3f}   MI = {mi:.2f}\n{verdict}", fontsize=9)
    ax.set_xlabel('x'); ax.set_ylabel('y')

plt.tight_layout()
plt.show()

print("Lesson: Pearson measures the *signed* linear trend — symmetric deviations cancel.")
print("MI sums the *absolute* density deviation from independence — no cancellation.")

## 2 · Method 1 — Univariate R²

**Question:** If I used only this feature, how much target variance would it explain?

**Shortcut:** For linear regression, univariate R² = ρ(xⱼ, y)². Read it from the correlation matrix — no need to fit 8 models.

In [ ]:
# Build a combined DataFrame with the target
df_train = X_train.copy()
df_train['MedHouseVal'] = y_train

corr_with_target = df_train.corr()['MedHouseVal'].drop('MedHouseVal')
univariate_r2 = (corr_with_target ** 2).sort_values(ascending=False)

print('Univariate R² (ρ² shortcut):')
print('-' * 40)
for feat, val in univariate_r2.items():
    bar = '█' * int(val * 80)
    print(f'  {feat:12s}  {val:.3f}  {bar}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#1d4ed8' if v > 0.05 else '#94a3b8' for v in univariate_r2.values]
bars = ax.barh(univariate_r2.index[::-1], univariate_r2.values[::-1], color=colors[::-1])
for bar, val in zip(bars, univariate_r2.values[::-1]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=9)
ax.set_xlabel('Univariate R²  (proportion of target variance explained alone)')
ax.set_title('Feature Importance — Method 1: Univariate R²\n'
             '"If I used only this feature, how much would I explain?"', fontsize=11)
ax.axvline(0.05, color='#ef4444', linestyle='--', alpha=0.6, label='5% threshold')
ax.legend(fontsize=9)
ax.set_xlim(0, 0.55)
plt.tight_layout()
plt.savefig('img/univariate_r2.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nObservation: MedInc towers over everything. Every other feature looks flat at this scale.')
print('This is NOT the full picture — it is the starting question.')

## 3 · Method 2 — Standardised Weights (Partial Contribution)

**Question:** Given all other features in the model, how much does each feature contribute?

This is the *partial* effect — what each feature adds above and beyond everything else.

In [ ]:
std_weights = pd.Series(
    model.coef_,
    index=housing.feature_names
)
abs_weights = std_weights.abs().sort_values(ascending=False)

print('Standardised weights (|wⱼ|) — partial contribution:')
print('-' * 50)
for feat, val in abs_weights.items():
    sign = '+' if std_weights[feat] >= 0 else '−'
    bar = '█' * int(val * 10)
    print(f'  {feat:12s}  {sign}{val:.3f}  {bar}')

print('\nNote: Latitude and Longitude are now at the TOP — they were near the BOTTOM in univariate R².')
print('This reversal is the key teaching moment.')

## 4 · The Surprise — Why the Rankings Diverge

Let's directly visualise the ranking gap between the two methods.

In [ ]:
# Normalise both to 0-1 scale for visual comparison
uni_norm = univariate_r2 / univariate_r2.max()
wt_norm  = abs_weights   / abs_weights.max()

comparison = pd.DataFrame({
    'Univariate R² (alone)':      uni_norm,
    'Std weight (joint model)': wt_norm
}).sort_values('Std weight (joint model)', ascending=False)

fig, ax = plt.subplots(figsize=(10, 4.5))
x = np.arange(len(comparison))
w = 0.35
ax.bar(x - w/2, comparison['Univariate R² (alone)'],      w, label='Univariate R² (alone)',    color='#94a3b8', alpha=0.85)
ax.bar(x + w/2, comparison['Std weight (joint model)'], w, label='Std weight (joint)',  color='#1d4ed8', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(comparison.index, rotation=20, ha='right')
ax.set_ylabel('Normalised importance (1 = max)')
ax.set_title('Where the Two Methods Agree — and Where They Diverge\n'
             'Blue bars rising vs grey: jointly irreplaceable (Lat, Lon)\n'
             'Grey bars falling vs blue: signal is shared with others (AveRooms)', fontsize=10)
ax.legend()
plt.tight_layout()
plt.savefig('img/ranking_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nKey divergences:')
print('  Lat/Lon:   low univariate → high joint  → JOINTLY IRREPLACEABLE')
print('  AveRooms:  moderate univariate → lower joint → SIGNAL SHARED with AveBedrms')

## 5 · Feature Correlation Heatmap

Visualising *feature × feature* correlations reveals the collinear pairs **before** computing VIF.

In [ ]:
corr_matrix = X_train.corr()

fig, ax = plt.subplots(figsize=(8, 6.5))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)  # upper triangle only
sns.heatmap(
    corr_matrix,
    annot=True, fmt='.2f', cmap='coolwarm', center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    ax=ax,
    annot_kws={'size': 8}
)
ax.set_title('Feature Correlation Heatmap\n'
             'High |ρ| between features = collinearity risk\n'
             'Red circles: AveRooms/AveBedrms (ρ≈0.85) and Lat/Lon (ρ≈−0.92)', fontsize=10)
# Highlight the two collinear pairs with rectangles
feat_names = list(corr_matrix.columns)
for pair in [('AveRooms', 'AveBedrms'), ('Latitude', 'Longitude')]:
    i, j = feat_names.index(pair[0]), feat_names.index(pair[1])
    for (r, c) in [(i, j), (j, i)]:
        ax.add_patch(plt.Rectangle((c, r), 1, 1, fill=False,
                                    edgecolor='black', lw=2.5))
plt.tight_layout()
plt.savefig('img/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"AveRooms / AveBedrms correlation: {corr_matrix.loc['AveRooms','AveBedrms']:.2f}")
print(f"Latitude / Longitude  correlation: {corr_matrix.loc['Latitude','Longitude']:.2f}")

## 6 · VIF — Quantifying Multicollinearity

VIF measures how much each feature's weight is inflated by its correlation with the **other features**.

$$\text{VIF}_j = \frac{1}{1 - R^2_j}$$

where $R^2_j$ is the R² from regressing feature $j$ on **all other features** (not the target).

In [ ]:
vif_df = pd.DataFrame({
    'Feature': housing.feature_names,
    'VIF': [variance_inflation_factor(X_train.values, i)
            for i in range(X_train.shape[1])]
}).sort_values('VIF', ascending=False).reset_index(drop=True)

def vif_flag(v):
    if v > 10: return '❌  SEVERE'
    if v > 5:  return '⚡  HIGH'
    if v > 2:  return '⚠️   Moderate'
    return '✅  OK'

vif_df['Status'] = vif_df['VIF'].apply(vif_flag)
print('VIF Analysis:')
print(vif_df.to_string(index=False))

print('\nInterpretation:')
print('  VIF = 7.2 on AveRooms means its standard error is √7.2 ≈ 2.7× larger than')
print('  it would be if AveRooms were uncorrelated with all other features.')
print('  The weight estimate is noisy — not necessarily wrong, but unreliable for interpretation.')

In [ ]:
# Demonstrate weight instability under multicollinearity
print('Demonstrating weight instability (AveRooms vs AveBedrms):')
print('──────────────────────────────────────────────────────────')
for seed in [42, 7, 99, 1234, 2025]:
    _, X_sub, _, y_sub = train_test_split(
        X_train, y_train, test_size=0.5, random_state=seed
    )
    scaler_sub = StandardScaler()
    X_sub_s = scaler_sub.fit_transform(X_sub)
    m = LinearRegression().fit(X_sub_s, y_sub)
    idx_rooms    = housing.feature_names.tolist().index('AveRooms')
    idx_bedrms   = housing.feature_names.tolist().index('AveBedrms')
    w_rooms  = m.coef_[idx_rooms]
    w_bedrms = m.coef_[idx_bedrms]
    print(f'  seed={seed:4d}  AveRooms={w_rooms:+.3f}  AveBedrms={w_bedrms:+.3f}')

print('\nThe weights flip sign and magnitude across subsets.')
print('Predictions on the full test set are nearly identical despite this — VIF in action.')

## 7 · Method 3 — Permutation Importance

**Question:** If I *scramble* this feature on the test set (destroying its signal), how much does MAE rise?

This is the **most reliable** and model-agnostic method. Crucially, the model is never retrained — you're measuring how badly the model's existing weights are handicapped when a feature's signal is destroyed. This makes it a pure test of the model's *reliance* on each feature, not just correlation or fitted weights.

In [ ]:
perm = permutation_importance(
    model, X_test_s, y_test,
    n_repeats=30,
    scoring='neg_mean_absolute_error',
    random_state=42
)
perm_imp = pd.Series(
    perm.importances_mean, index=housing.feature_names
).sort_values(ascending=False)
perm_std = pd.Series(perm.importances_std, index=housing.feature_names)

print('Permutation importance (mean Δ MAE when feature is shuffled):')
print('-' * 60)
for feat in perm_imp.index:
    val = perm_imp[feat]
    std = perm_std[feat]
    delta_usd = val * 100_000
    print(f'  {feat:12s}  {val:.4f} ± {std:.4f}  (≈${delta_usd:+,.0f} MAE rise if shuffled)')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#1d4ed8' if v > 0.01 else '#94a3b8' for v in perm_imp.values]
ax.barh(
    perm_imp.index[::-1], perm_imp.values[::-1],
    xerr=perm_std[perm_imp.index[::-1]].values,
    color=colors[::-1], alpha=0.85, capsize=3
)
ax.set_xlabel('Mean Δ MAE (neg_mean_absolute_error) when feature is shuffled')
ax.set_title('Feature Importance — Method 3: Permutation Importance\n'
             'Error bars = std across 30 repeats', fontsize=11)
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig('img/permutation_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 7b · Permutation Shuffle Loop — Visualizing How It Works

Let's demonstrate the shuffle loop on a small sample to show exactly what permutation importance measures. We'll take 10 test samples, shuffle one feature (MedInc), and show that predictions change even though the model weights stay frozen.

In [ ]:
# Take a small sample from test set for visualization
n_samples = 10
sample_idx = np.random.choice(len(X_test_s), n_samples, replace=False)
X_sample = X_test_s[sample_idx].copy()
y_sample = y_test[sample_idx]

# Baseline predictions
y_pred_baseline = model.predict(X_sample)
mae_baseline = mean_absolute_error(y_sample, y_pred_baseline)

# Shuffle MedInc column (feature index 0)
X_sample_shuffled = X_sample.copy()
medinc_idx = list(housing.feature_names).index('MedInc')
np.random.shuffle(X_sample_shuffled[:, medinc_idx])  # shuffle in-place

# Predictions after shuffle (model weights UNCHANGED)
y_pred_shuffled = model.predict(X_sample_shuffled)
mae_shuffled = mean_absolute_error(y_sample, y_pred_shuffled)

print("Permutation Shuffle Loop — Step-by-Step Demo")
print("=" * 70)
print(f"Step 1 — Baseline predictions on {n_samples} test samples:")
print(f"  MAE = ${mae_baseline * 100_000:,.0f}")
print(f"\nStep 2 — Shuffle MedInc column (break the income signal):")
print(f"  Original MedInc (scaled): {X_sample[:, medinc_idx][:5].round(2)}")
print(f"  Shuffled MedInc (scaled): {X_sample_shuffled[:, medinc_idx][:5].round(2)}")
print(f"  ⚠️  Model weights are FROZEN — we're not retraining!")
print(f"\nStep 3 — Predict with shuffled feature:")
print(f"  MAE = ${mae_shuffled * 100_000:,.0f}  (worse!)")
print(f"\nStep 4 — Compute importance:")
print(f"  Δ MAE = ${(mae_shuffled - mae_baseline) * 100_000:,.0f}")
print(f"  → MedInc permutation importance ≈ {(mae_shuffled - mae_baseline):.3f}")
print("=" * 70)

# 4-panel visualization
fig = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.3)

# Panel 1: Baseline predictions
ax1 = fig.add_subplot(gs[0, 0])
ax1.scatter(range(n_samples), y_sample * 100_000, label='Actual', s=80, alpha=0.7, color='black', marker='o')
ax1.scatter(range(n_samples), y_pred_baseline * 100_000, label='Predicted', s=80, alpha=0.7, color='#10b981', marker='x')
ax1.set_ylabel('House Value ($)')
ax1.set_xlabel('Sample index')
ax1.set_title(f'Panel 1: Baseline Predictions\nMAE = ${mae_baseline * 100_000:,.0f}', fontsize=11, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# Panel 2: Feature shuffle visualization
ax2 = fig.add_subplot(gs[0, 1])
x_pos = np.arange(n_samples)
ax2.bar(x_pos - 0.2, X_sample[:, medinc_idx], width=0.4, label='Original MedInc', color='#3b82f6', alpha=0.7)
ax2.bar(x_pos + 0.2, X_sample_shuffled[:, medinc_idx], width=0.4, label='Shuffled MedInc', color='#ef4444', alpha=0.7)
ax2.set_ylabel('MedInc (standardized)')
ax2.set_xlabel('Sample index')
ax2.set_title('Panel 2: MedInc Column Shuffled\n⚠️ Signal destroyed, model weights unchanged', fontsize=11, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

# Panel 3: Predictions after shuffle
ax3 = fig.add_subplot(gs[1, 0])
ax3.scatter(range(n_samples), y_sample * 100_000, label='Actual', s=80, alpha=0.7, color='black', marker='o')
ax3.scatter(range(n_samples), y_pred_shuffled * 100_000, label='Predicted (shuffled)', s=80, alpha=0.7, color='#ef4444', marker='x')
ax3.set_ylabel('House Value ($)')
ax3.set_xlabel('Sample index')
ax3.set_title(f'Panel 3: Predictions After Shuffle\nMAE = ${mae_shuffled * 100_000:,.0f} (worse!)', fontsize=11, fontweight='bold', color='#ef4444')
ax3.legend()
ax3.grid(alpha=0.3)

# Panel 4: Delta MAE
ax4 = fig.add_subplot(gs[1, 1])
labels = ['Baseline\nMAE', 'Shuffled\nMAE', 'Δ MAE\n(importance)']
values = [mae_baseline * 100_000, mae_shuffled * 100_000, (mae_shuffled - mae_baseline) * 100_000]
colors_bar = ['#10b981', '#ef4444', '#f59e0b']
bars = ax4.bar(labels, values, color=colors_bar, alpha=0.8, edgecolor='black', linewidth=1.5)
for bar, val in zip(bars, values):
    ax4.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1000,
             f'${val:,.0f}', ha='center', fontsize=10, fontweight='bold')
ax4.set_ylabel('MAE ($)')
ax4.set_title('Panel 4: Permutation Importance\nΔ MAE = how much model relies on MedInc', fontsize=11, fontweight='bold')
ax4.set_ylim(0, max(values) * 1.15)

fig.suptitle('Permutation Importance — The Shuffle Loop in Action', fontsize=14, fontweight='bold')
plt.show()

print("\n✅ Key insight: The model is never retrained.")
print("   We measure how badly existing weights are handicapped when a signal is destroyed.")
print(f"   On the full test set, MedInc permutation importance = {perm_imp['MedInc']:.3f}")

## 8 · Three-View Dashboard

Side-by-side comparison of all three methods. Features where rankings diverge tell the richest story.

In [ ]:
dashboard = pd.DataFrame({
    'Univariate R²':    univariate_r2,
    '|Std weight|':     abs_weights,
    'Perm importance':  perm_imp
})

# Rank each column (1 = most important)
ranks = dashboard.rank(ascending=False).astype(int)
ranks.columns = ['Rank (uni R²)', 'Rank (std wt)', 'Rank (perm)']

full_dash = pd.concat([dashboard.round(3), ranks], axis=1)

def verdict(row):
    r = (row['Rank (uni R²)'], row['Rank (std wt)'], row['Rank (perm)'])
    if max(r) <= 3:
        return '✅ Strong independent signal'
    if r[0] >= 5 and r[1] <= 3:
        return '🔵 Jointly irreplaceable'
    if row['Perm importance'] < 0.005:
        return '❌ Near-zero contribution'
    if r[1] >= 6 and row['|Std weight|'] < 0.05:
        return '⚡ Collinear — weight unstable'
    return '⚠️  Modest'

full_dash['Verdict'] = full_dash.apply(verdict, axis=1)
print(full_dash.to_string())

In [ ]:
# Visual three-panel summary
fig = plt.figure(figsize=(15, 4.5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.45)

methods = [
    ('Univariate R²\n(alone)', univariate_r2, '#94a3b8'),
    ('Std |weight|\n(joint)', abs_weights, '#1d4ed8'),
    ('Permutation\n(test set)', perm_imp, '#15803d'),
]
for idx, (title, data, color) in enumerate(methods):
    ax = fig.add_subplot(gs[0, idx])
    order = data.sort_values(ascending=True)
    ax.barh(order.index, order.values, color=color, alpha=0.85)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('Importance')
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

fig.suptitle('Three-View Feature Importance Dashboard — California Housing',
             fontsize=12, fontweight='bold', y=1.02)
plt.savefig('img/three_view_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: img/three_view_dashboard.png')

## 9 · Multicollinearity Deep Dive — AveRooms vs AveBedrms

Visualise *why* the two room features cause weight instability.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scatter: AveRooms vs AveBedrms
axes[0].scatter(
    X_train['AveRooms'].clip(0, 10),
    X_train['AveBedrms'].clip(0, 5),
    alpha=0.05, s=4, color='#1d4ed8'
)
rho = X_train['AveRooms'].corr(X_train['AveBedrms'])
axes[0].set_xlabel('AveRooms (clipped at 10)')
axes[0].set_ylabel('AveBedrms (clipped at 5)')
axes[0].set_title(f'AveRooms vs AveBedrms\nρ = {rho:.2f}  (highly collinear)\n'
                  'The model cannot cleanly separate their contributions.', fontsize=9)

# Weight instability across bootstrap sub-samples
rooms_wts, bedrms_wts = [], []
idx_r = housing.feature_names.tolist().index('AveRooms')
idx_b = housing.feature_names.tolist().index('AveBedrms')
for seed in range(100):
    _, X_sub, _, y_sub = train_test_split(
        X_train, y_train, test_size=0.3, random_state=seed
    )
    sc = StandardScaler()
    Xs = sc.fit_transform(X_sub)
    c  = LinearRegression().fit(Xs, y_sub).coef_
    rooms_wts.append(c[idx_r])
    bedrms_wts.append(c[idx_b])

axes[1].scatter(rooms_wts, bedrms_wts, alpha=0.5, s=20, color='#ef4444')
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].axvline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_xlabel('AveRooms weight (100 bootstrap sub-samples)')
axes[1].set_ylabel('AveBedrms weight')
axes[1].set_title('Weight instability across sub-samples\n'
                  'Each dot = one 70% sub-sample. Scatter = unstable estimates.\n'
                  'Predictions are fine; individual weights are not.', fontsize=9)

plt.tight_layout()
plt.savefig('img/collinearity_demo.png', dpi=150, bbox_inches='tight')
plt.show()

## 9b · Joint Feature Importance — Cooperation vs Competition

VIF catches the *competition* case: two features measure the same thing and their weights blow up. The opposite exists too: two features can be **individually weak but jointly irreplaceable** — the *cooperation* case.

**Diagnostic — joint permutation importance**: shuffle both features simultaneously and compare the drop to the sum of their individual drops.

$$\Delta_{\text{interact}}(j,k) = \pi_{jk} - \pi_j - \pi_k$$

- $\Delta > 0$ → the model is using their *interaction*; the features cooperate (Lat/Long case)
- $\Delta \approx 0$ → additive independence
- $\Delta < 0$ → the features are substitutes; one is largely redundant (AveRooms/AveBedrms case)

In [ ]:
def joint_perm_importance(model, X, y, feat_a_idx, feat_b_idx, n_repeats=20, random_state=42):
    """Shuffle feat_a and feat_b simultaneously and return mean Δ MAE."""
    rng = np.random.default_rng(random_state)
    baseline = mean_absolute_error(y, model.predict(X))
    deltas = []
    for _ in range(n_repeats):
        X_shuf = X.copy()
        perm = rng.permutation(len(X_shuf))
        X_shuf[:, feat_a_idx] = X_shuf[perm, feat_a_idx]
        X_shuf[:, feat_b_idx] = X_shuf[perm, feat_b_idx]   # same permutation → spatial pair destroyed
        deltas.append(mean_absolute_error(y, model.predict(X_shuf)) - baseline)
    return np.mean(deltas)

feat = housing.feature_names.tolist()
idx_lat  = feat.index('Latitude')
idx_lon  = feat.index('Longitude')
idx_room = feat.index('AveRooms')
idx_bed  = feat.index('AveBedrms')

# Individual permutation importances (already computed above — grab from perm_imp)
pi_lat  = perm_imp['Latitude']
pi_lon  = perm_imp['Longitude']
pi_room = perm_imp['AveRooms']
pi_bed  = perm_imp['AveBedrms']

# Joint permutation importance
pi_lat_lon  = joint_perm_importance(model, X_test_s, y_test, idx_lat,  idx_lon,  n_repeats=20)
pi_room_bed = joint_perm_importance(model, X_test_s, y_test, idx_room, idx_bed,  n_repeats=20)

print("Joint Permutation Importance Analysis")
print("=" * 60)
print("\nCase 1 — Latitude + Longitude (cooperation expected):")
print(f"  π_Lat  alone          = {pi_lat:.4f}  (${pi_lat*100_000:,.0f} MAE rise)")
print(f"  π_Long alone          = {pi_lon:.4f}  (${pi_lon*100_000:,.0f} MAE rise)")
print(f"  Sum of individuals    = {pi_lat+pi_lon:.4f}  (${(pi_lat+pi_lon)*100_000:,.0f})")
print(f"  π_{{Lat+Long}} joint    = {pi_lat_lon:.4f}  (${pi_lat_lon*100_000:,.0f} MAE rise)")
uplift_geo = pi_lat_lon - (pi_lat + pi_lon)
print(f"  Interaction uplift Δ  = {uplift_geo:+.4f}  (${uplift_geo*100_000:+,.0f})  ← {'COOPERATION' if uplift_geo > 0 else 'additive'}")

print("\nCase 2 — AveRooms + AveBedrms (competition/redundancy expected):")
print(f"  π_AveRooms alone      = {pi_room:.4f}  (${pi_room*100_000:,.0f} MAE rise)")
print(f"  π_AveBedrms alone     = {pi_bed:.4f}  (${pi_bed*100_000:,.0f} MAE rise)")
print(f"  Sum of individuals    = {pi_room+pi_bed:.4f}  (${(pi_room+pi_bed)*100_000:,.0f})")
print(f"  π_{{Rooms+Beds}} joint  = {pi_room_bed:.4f}  (${pi_room_bed*100_000:,.0f} MAE rise)")
uplift_room = pi_room_bed - (pi_room + pi_bed)
print(f"  Interaction uplift Δ  = {uplift_room:+.4f}  (${uplift_room*100_000:+,.0f})  ← {'COOPERATION' if uplift_room > 0 else 'substitutes / redundant'}")

print("\nConclusion:")
print("  Lat+Long: joint loss > sum → model exploits the *position* (2D), not just axes → interaction feature candidate")
print("  Rooms+Beds: joint loss ≈ individual → one is redundant → drop AveBedrms in Ch.4/5")

# Visualize the comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Joint Permutation Importance: Cooperation vs Competition", fontsize=12, fontweight='bold')

for ax, (label, pi_a, pi_b, pi_ab, name_a, name_b) in zip(axes, [
    ("Lat + Long\n(cooperation)", pi_lat, pi_lon, pi_lat_lon, "Latitude", "Longitude"),
    ("Rooms + Beds\n(substitutes)", pi_room, pi_bed, pi_room_bed, "AveRooms", "AveBedrms"),
]):
    vals = [pi_a, pi_b, pi_a + pi_b, pi_ab]
    labels = [f'π_{name_a}', f'π_{name_b}', 'Sum\n(independent)', 'Joint\nπ_AB']
    colors = ['#94a3b8', '#94a3b8', '#f59e0b', '#1d4ed8']
    bars = ax.bar(labels, vals, color=colors, alpha=0.85, edgecolor='black')
    ax.set_ylabel('Δ MAE (importance)')
    ax.set_title(label, fontsize=10)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.02,
                f'{v:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## 10 · Action Items for Ch.4 & Ch.5

Based on the three-view dashboard, what do we know about pursuing the $40k MAE target?

In [ ]:
print('═' * 65)
print(' FEATURE IMPORTANCE AUDIT — ACTION ITEMS')
print('═' * 65)

print('''
Ch.4 · Polynomial Features (target: ~$48k MAE)
───────────────────────────────────────────────
✅ HIGH-VALUE polynomial targets (strong permutation importance):
   • MedInc²  — income has a known non-linear ceiling effect
   • MedInc × Latitude  — coastal premium (geographic income signal)
   • MedInc × Longitude — East/West income gradient
   • Latitude × Longitude — geographic cluster interactions

⚠️  CAUTIOUS additions (low permutation importance):
   • AveBedrms² — very low permutation importance; adds noise risk
   • Population² — near-zero contribution; don't expand

Ch.5 · Regularization (target: ~$38k MAE)
──────────────────────────────────────────
⚡ Collinear pair to handle with Ridge:
   • AveRooms / AveBedrms (VIF ≈ 7)  → Ridge shrinks toward shared signal
   • Alternative: drop AveBedrms, or create rooms_per_bedroom feature

⚡ Correlated pair (informative but anti-correlated):
   • Latitude / Longitude (VIF ≈ 3.5) → keep both; they are jointly irreplaceable
   • Ridge safely handles their moderate VIF

❌ Feature to potentially drop:
   • Population — permutation importance ≈ 0; adds noise, costs a parameter
''')

print('═' * 65)

## Summary

| Method | Top feature | Bottom feature | Best use |
|---|---|---|---|
| Univariate R² | MedInc (0.47) | Population (0.001) | First-pass scan; quick ranking |
| Std \|weight\| | Latitude (0.89) | Population (0.01) | Final model inspection; requires standardisation |
| Permutation | MedInc (+$18k) | Population (+$0.1k) | Most reliable; model-agnostic; use on test set |

**Key takeaways:**
1. **MedInc** is the strongest individual predictor regardless of method.
2. **Latitude and Longitude** are jointly irreplaceable — low alone, high together.
3. **AveRooms / AveBedrms** are collinear (VIF ≈ 7) — their weight split is arbitrary.
4. **Population** contributes virtually nothing and is a candidate for removal.
5. Always inspect all three views; a feature cheap in one can be critical in another.

**Next:** Ch.4 — Polynomial Features: add `MedInc²`, `MedInc × Latitude`, `MedInc × Longitude` to push MAE toward $48k.